<a href="https://colab.research.google.com/github/ryueuitae-blip/AIEYES-STUDY/blob/suyeon_branch/6%EC%A3%BC%EC%B0%A8_%EC%97%B0%EB%B4%89_%EC%98%88%EC%B8%A1_%EB%B6%84%EC%84%9D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 프로야구 연봉 데이터 파일 경로 지정
picher_file_path = '../data/picher_stats_2017.csv'
batter_file_path = '../data/batter_stats_2017.csv'

# CSV 파일 불러오기
picher = pd.read_csv(picher_file_path)
batter = pd.read_csv(batter_file_path)

FileNotFoundError: [Errno 2] No such file or directory: '../data/picher_stats_2017.csv'

In [ ]:
picher.columns

In [ ]:
picher.head()

In [ ]:
print(picher.shape)

In [ ]:
# 한글 폰트 설치

!apt-get install -y fonts-nanum

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  fonts-nanum
0 upgraded, 1 newly installed, 0 to remove and 3 not upgraded.
Need to get 10.3 MB of archives.
After this operation, 34.1 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 fonts-nanum all 20200506-1 [10.3 MB]
Fetched 10.3 MB in 0s (48.3 MB/s)
Selecting previously unselected package fonts-nanum.
(Reading database ... 118252 files and directories currently installed.)
Preparing to unpack .../fonts-nanum_20200506-1_all.deb ...
Unpacking fonts-nanum (20200506-1) ...
Setting up fonts-nanum (20200506-1) ...
Processing triggers for fontconfig (2.13.1-4.2ubuntu5) ...


In [ ]:
import matplotlib.pyplot as plt

plt.rc('font', family='NanumBarunGothic')

In [ ]:
# 2018년 연봉 분포(histogram) 출력
picher['연봉(2018)'].hist(bins=100)

In [ ]:
# 연봉 데이터 boxplot 출력
picher.boxplot(column=['연봉(2018)'])

회귀분석에 사용할 피처 살펴보기

In [ ]:
# 회귀 분석에 사용할 피처(특징)만 추출
picher_features_df = picher[
    ['승', '패', '세', '홀드', '블론', '경기', '선발', '이닝',
     '삼진/9', '볼넷/9', '홈런/9', 'BABIP', 'LOB%', 'ERA',
     'RA9-WAR', 'FIP', 'kFIP', 'WAR',
     '연봉(2018)', '연봉(2017)']
]

NameError: name 'picher' is not defined

In [ ]:
# 각 피처별 histogram(분포 그래프) 출력 함수
def plot_hist_each_column(df):

    plt.rcParams['figure.figsize'] = [20, 16] # 그래프 크기 설정
    fig = plt.figure(1)
    for i in range(len(df.columns)): # 모든 컬럼에 대해 반복하며 subplot 생성
        ax = fig.add_subplot(5, 5, i+1) # 5x5 형태의 subplot 생성
        plt.hist(df[df.columns[i]], bins=50) # 각 컬럼의 히스토그램 출력
        ax.set_title(df.columns[i]) # 그래프 제목을 컬럼명으로 설정

    plt.show()

In [ ]:
# 피처별 분포 그래프 출력
plot_hist_each_column(picher_features_df)

투수의 연봉 예측하기

1. 피처들의 단위 맞춰주기: 피처 스케일링

In [ ]:
# pandas 출력 시 일반 float 형태로 출력
pd.options.mode.chained_assignment = None

In [ ]:
# 각 피처(feature)에 대해 scaling(표준화) 수행
def standard_scaling(df, scale_columns):
    for col in scale_columns:

        # 스케일링할 컬럼의 평균과 표준편차 계산
        series_mean = df[col].mean()
        series_std = df[col].std()

        # (값 - 평균) / 표준편차 방식으로 표준화 수행
        df[col] = df[col].apply(lambda x: (x - series_mean) / series_std)

    return df

In [ ]:
# 스케일링을 적용할 컬럼 지정
scale_columns = ['승', '패', '세', '홀드', '블론', '경기', '선발', '이닝', '삼진/9',
    '볼넷/9', '홈런/9', 'BABIP', 'LOB%', 'ERA', 'RA9-WAR', 'FIP', 'kFIP', 'WAR', '연봉(2017)']

picher_df = standard_scaling(picher, scale_columns) # 표준화 적용

In [ ]:
# 예측 대상 컬럼 이름을 y로 변경
picher_df = picher_df.rename(columns={'연봉(2018)': 'y'})

picher_df.head(5) # 상위 5개 출력

2. 피처들의 단위 맞춰주기: one-hot-encoding

In [ ]:
# 팀명 피처를 one-hot encoding 형태로 변환
team_encoding = pd.get_dummies(picher_df['팀명'])

picher_df = picher_df.drop('팀명', axis=1) # 기존 팀명 컬럼은 제거

# one-hot encoding 결과를 기존 데이터프레임에 추가
picher_df = picher_df.join(team_encoding)

In [ ]:
team_encoding.head(5)

In [ ]:
picher_df.head()

회귀분석을 위해 학습, 테스트 데이터셋 분리

In [ ]:
from sklearn import linear_model
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from math import sqrt

# 학습 데이터와 테스트 데이터를 분리
# X : 입력 데이터(연봉 제외한 피처들) / y : 예측 대상 데이터(2018 연봉)
X = picher_df[picher_df.columns.difference(['선수명', 'y'])]
y = picher_df['y']

# train 데이터 80%, test 데이터 20%로 분리
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,   # 테스트 데이터 비율
    random_state=19  # 결과 고정
)

In [ ]:
X_train.shape

In [ ]:
y_train.shape

In [ ]:
X_test.shape

In [ ]:
y_test.shape

회귀분석 계수 학습 & 학습된 계수 출력

In [ ]:
lr = linear_model.LinearRegression() # 선형 회귀 모델 생성
model = lr.fit(X_train, y_train) # 학습 데이터로 회귀 모델 학습

In [ ]:
# 학습된 회귀 계수(coef) 출력: 각 피처가 연봉 예측에 얼마나 영향을 주는지 확인 가능
print(lr.coef_)

예측 모델 평가하기

어떤 피처가 가장 영향력이 강할까?

In [ ]:
import statsmodels.api as sm # statsmodels 라이브러리를 이용해 회귀 분석을 수행

# 상수항(constant) 추가: 회귀식의 절편 값을 계산하기 위해 필요함
X_train = sm.add_constant(X_train)

# OLS(최소제곱법) 선형 회귀 모델 학습
model = sm.OLS(y_train, X_train).fit()

model.summary() # 회귀 분석 결과 요약 출력

예측모델의 평가

In [ ]:
# 학습 데이터와 테스트 데이터를 다시 분리
# X : 입력 데이터(선수명, 연봉 제외) / y : 예측 대상 데이터(2018 연봉)
X = picher_df[picher_df.columns.difference(['선수명', 'y'])]
y = picher_df['y']

# train 데이터 80%, test 데이터 20%로 분리
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=19
)

In [ ]:
# 선형 회귀 모델 생성 및 학습
lr = linear_model.LinearRegression()

model = lr.fit(X_train, y_train) # train 데이터로 모델 학습

R2 score

In [ ]:
# train 데이터의 R2 score 출력 (학습 데이터 설명력)
print(model.score(X_train, y_train))

# test 데이터의 R2 score 출력 (새 데이터 예측 성능)
print(model.score(X_test, y_test))

RMSE score

In [ ]:
# train 데이터에 대한 예측 수행
y_predictions = lr.predict(X_train)

# train RMSE score 출력: 값이 작을수록 예측 오차가 적음
print(sqrt(mean_squared_error(y_train, y_predictions)))

# test 데이터에 대한 예측 수행
y_predictions = lr.predict(X_test)

# test RMSE score 출력
print(sqrt(mean_squared_error(y_test, y_predictions)))

피처들의 상관관계 분석

In [ ]:
import seaborn as sns

# 피처들 간의 상관계수 계산
corr = picher_df[scale_columns].corr(method='pearson')

show_cols = ['win', 'lose', 'save', 'hold', 'blon', 'match',
             'start', 'inning', 'strike', 'ball4', 'homerun',
             'BABIP', 'LOB', 'ERA', 'RA9-WAR', 'FIP',
             'kFIP', 'WAR', '2017']

sns.set(font_scale=1.5) # 상관관계 히트맵 출력

hm = sns.heatmap(
    corr.values,
    cbar=True,
    annot=True,   # 상관계수 값 표시
    square=True,
    fmt='.2f',
    annot_kws={'size': 15},
    yticklabels=show_cols,
    xticklabels=show_cols
)

plt.tight_layout()
plt.show()

회귀분석 예측 성능을 높이기 위한 방법: 다중공선성 확인

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [ ]:
# 피처들의 VIF 값 계산 -> 다중공선성 확인

vif = pd.DataFrame()

vif["VIF Factor"] = [variance_inflation_factor(X.values, i)
    for i in range(X.shape[1])]

vif["features"] = X.columns
vif.round(1) # 소수점 1자리까지 출력

적절한 피처로 다시 학습하기

In [ ]:
# 선형 회귀 모델 학습
lr = linear_model.LinearRegression()

model = lr.fit(X_train, y_train)

In [ ]:
# R2 score 출력
print(model.score(X_train, y_train))
print(model.score(X_test, y_test))

In [ ]:
# RMSE 계산
y_predictions = lr.predict(X_train)

print(sqrt(mean_squared_error(y_train, y_predictions)))

y_predictions = lr.predict(X_test)

print(sqrt(mean_squared_error(y_test, y_predictions)))

In [ ]:
# 선택한 피처들의 VIF 확인
X = picher_df[['FIP', 'WAR', '볼넷/9', '삼진/9', '연봉(2017)']]

vif = pd.DataFrame()

vif["VIF Factor"] = [variance_inflation_factor(X.values, i)
    for i in range(X.shape[1])]

vif["features"] = X.columns
vif.round(1)

예상 연봉과 실제 연봉 비교

In [ ]:
# 2018년 연봉 예측값 생성
X = picher_df[['FIP', 'WAR', '볼넷/9', '삼진/9', '연봉(2017)']]
predict_2018_salary = lr.predict(X)
picher_df['예측연봉(2018)'] = pd.Series(predict_2018_salary)

In [ ]:
# 원본 데이터 다시 불러오기
picher = pd.read_csv(picher_file_path)
picher = picher[['선수명', '연봉(2017)']]

# 실제 연봉 데이터 합치기
result_df = picher_df.sort_values(by=['y'], ascending=False)
result_df.drop(['연봉(2017)'], axis=1, inplace=True, errors='ignore')
result_df = result_df.merge(picher, on=['선수명'], how='left')

# 필요한 컬럼만 선택
result_df = result_df[['선수명', 'y', '예측연봉(2018)', '연봉(2017)']]

# 컬럼명 변경
result_df.columns = ['선수명', '실제연봉(2018)', '예측연봉(2018)', '작년연봉(2017)']

# 연봉이 변한 선수만 출력
result_df = result_df[result_df['작년연봉(2017)'] != result_df['실제연봉(2018)']]

result_df = result_df.reset_index()
result_df = result_df.iloc[:10, :]
result_df.head(10)

In [ ]:
# 연봉 비교 bar 그래프 출력
plt.rc('font', family='NanumGothicOTF')

result_df.plot(x='선수명', y=['작년연봉(2017)', '예측연봉(2018)', '실제연봉(2018)'], kind='bar', figsize=(12, 6))

plt.title('예상 연봉과 실제 연봉 비교')
plt.ylabel('연봉')
plt.show()